Nome del gruppo

In [12]:
import numpy as np
import pandas as pd
from keras.models import Sequential
from keras.layers import Dense, Dropout
import keras_tuner
import keras

import matplotlib as mpl
import matplotlib.pyplot as plt
# default font
plt.rcParams['font.size'] = 13

from sklearn.preprocessing import StandardScaler

%run useful.py

# training/validation/test fractions
perc_train = 0.7
perc_valid = 0.15
perc_test = 0.15
#  check it sums to 1 
assert abs(perc_train + perc_valid + perc_test - 1.0) < 1e-12

In [13]:
# Keras works with numpy arrays: just use them from the start

TYPE=4
# number of features per sample
L=8
# span of each component
B=10
x = np.loadtxt(filename("data",L,TYPE), delimiter=' ')
y = np.loadtxt(filename("labels",L,TYPE), delimiter=' ')
y = y.astype("int")
print(x.shape)
N = len(x)


# dim. of a sample
L = len(x[0])
print(L)

for i in range(5):
    print(x[i],y[i])

N_train = int(perc_train * N)
N_valid = int(perc_valid * N)
N_test = N - N_train - N_valid
print(f'data: {N}\ntrain: {N_train}\nvalid: {N_valid}\ntest: {N_test}')

(12000, 8)
8
[1.83918812 2.04560279 5.67725029 5.95544703 9.6451452  6.53177097
 7.48906638 6.53569871] 0
[7.47714809 9.61306736 0.08388298 1.06444377 2.98703714 6.56411183
 8.09812553 8.72175914] 0
[9.64647597 7.23685347 6.42475328 7.17453621 4.67599007 3.25584678
 4.39644606 7.29689083] 0
[9.94014586 6.76873712 7.90822518 1.70914258 0.26849276 8.00370244
 9.03722538 0.2467621 ] 0
[4.91747318 5.26255167 5.9636601  0.51957545 8.95089528 7.2826618
 8.18350011 5.00222753] 1
data: 12000
train: 8400
valid: 1800
test: 1800


In [14]:
def Standardize(x,m,s):
    """
    rescale each component using its mean and standard deviation
    x: data points (numpy array)
    m: mean values (numpy array)
    s: standard deviations (numpy array)
    return: rescaled data points (numpy array)
    """
    N = len(x)
    # assuming len(m)=len(s)=len(x[0])
    mm,ss = np.tile(m,(N,1)), np.tile(s,(N,1))
    return (x-mm)/ss

# split data into train/validation/test sets and check 
(x_train, y_train) = (x[0:N_train],y[0:N_train])
(x_valid, y_valid) = (x[N_train:N_train+N_valid],y[N_train:N_train+N_valid])
(x_test, y_test) = (x[N_train+N_valid:],y[N_train+N_valid:])
print("Train:",len(x_train),"\t Validation:",len(x_valid),"\t Test:",len(x_test))


x_train_mean = np.mean(x_train, axis=0)
x_train_std = np.std(x_train, axis=0)
#print("train stats before rescaling:\nmean value=", x_train_mean, "\nstd. dev.=", x_train_std)

x_valid_mean = np.mean(x_valid, axis=0)
x_valid_std = np.std(x_valid, axis=0)
#print("validation stats before rescaling:\nmean value=", x_valid_mean, "\nstd. dev.=", x_valid_std)

x_test_mean = np.mean(x_test, axis=0)
x_test_std = np.std(x_test, axis=0)
#print("test stats before rescaling:\nmean value=", x_test_mean, "\nstd. dev.=", x_test_std)

x_train = Standardize(x_train, x_train_mean, x_train_std)
x_valid = Standardize(x_valid, x_valid_mean, x_valid_std)
x_test = Standardize(x_test, x_test_mean, x_test_std)
print("after rescaling (train):\nmean value=", x_train.mean(axis=0), "\nstd. dev.=", x_train.std(axis=0))

Train: 8400 	 Validation: 1800 	 Test: 1800
after rescaling (train):
mean value= [ 9.17572896e-16  1.18834043e-14 -1.95478554e-15 -3.53337069e-15
 -6.42594443e-15 -2.40365928e-15 -2.88332932e-15 -8.27623684e-15] 
std. dev.= [1. 1. 1. 1. 1. 1. 1. 1.]


In [15]:
def build_model(hp):
    model = Sequential()
    
    # Input layer 
    model.add(Dense(L, input_shape=(L,), activation='relu'))
    
    # Search over dropout rate (same rate applied to all 3 dropout layers)
    dropout_rate = hp.Float("dropout_rate", min_value=0.0, max_value=0.4, step=0.1)
    activation = hp.Choice("activation", values=["relu", "elu"])
    # Inner layers (fixed at 3, same as your original)
    model.add(Dense(20, activation=activation))
    model.add(Dropout(dropout_rate))
    model.add(Dense(20, activation=activation))
    model.add(Dropout(dropout_rate))
    model.add(Dense(20, activation=activation))
    model.add(Dropout(dropout_rate))
    
    # Output layer 
    model.add(Dense(1, activation='sigmoid'))
    
    # Search over learning rate
    learning_rate = hp.Float("learning_rate", min_value=1e-6, max_value=1e-1, sampling="log")
    
    # Search over optimizer
    optimizer_name = hp.Choice("optimizer", values=["adam", "sgd", "rmsprop", "nadam"])
    
    # Map optimizer choice to actual optimizer with tuned learning rate
    if optimizer_name == "adam":
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == "sgd":
        optimizer = keras.optimizers.SGD(learning_rate=learning_rate)
    elif optimizer_name == "rmsprop":
        optimizer = keras.optimizers.RMSprop(learning_rate=learning_rate)
    elif optimizer_name == "nadam":
        optimizer = keras.optimizers.Nadam(learning_rate=learning_rate)
    
    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy", 
        metrics=["accuracy"]
    )
    
    return model

In [ ]:
tuner = keras_tuner.RandomSearch(
    hypermodel=build_model,
    objective="val_accuracy",
    max_trials=5,
    executions_per_trial=1,
    overwrite=True,
    directory="trials",
    project_name="my_search"
)

tuner.search(x_train, y_train,
             epochs=50,
             validation_data=(x_valid, y_valid),
             verbose=2)

Trial 1 Complete [00h 00m 40s]
val_accuracy: 0.5733333230018616

Best val_accuracy So Far: 0.5733333230018616
Total elapsed time: 00h 00m 40s

Search: Running Trial #2

Value             |Best Value So Far |Hyperparameter
0                 |0.1               |dropout_rate
relu              |elu               |activation
9.0141e-06        |0.00084633        |learning_rate
adam              |rmsprop           |optimizer

Epoch 1/50
263/263 - 3s - 10ms/step - accuracy: 0.5018 - loss: 0.6936 - val_accuracy: 0.5117 - val_loss: 0.6936
Epoch 2/50
263/263 - 1s - 3ms/step - accuracy: 0.5018 - loss: 0.6935 - val_accuracy: 0.5139 - val_loss: 0.6936
Epoch 3/50
263/263 - 1s - 3ms/step - accuracy: 0.5032 - loss: 0.6935 - val_accuracy: 0.5156 - val_loss: 0.6935
Epoch 4/50
263/263 - 1s - 3ms/step - accuracy: 0.5043 - loss: 0.6934 - val_accuracy: 0.5150 - val_loss: 0.6934
Epoch 5/50
263/263 - 1s - 3ms/step - accuracy: 0.5043 - loss: 0.6933 - val_accuracy: 0.5144 - val_loss: 0.6934
Epoch 6/50
263/263 - 

In [ ]:
#takes as parameters a list of models, the datasaet
#and the number of k-folds
#gives back the best model index in the list and
#the list of errors for the models
def cross_validation(models, x, y, k=5)->tuple:
    import numpy as np

    #gives back the cost for the given model
    def fit(model, x_train, y_train, x_val, y_val):
        model.fit(x, y)
        y_pred = model.predict(x_val)
        # Return a single float to minimize.
        return np.mean((y_pred - y_val)**2)

    #shuffling the indexes for shuflling the dataset
    #before dividing it in k folds
    idxs = np.arange(y.size)
    np.random.shuffle(idxs)

    #creating folds
    x_fold = []
    y_fold = []

    for i in range(k):
        x_fold.append(x[idxs[i::k]])
        y_fold.append(y[idxs[i::k]])

    #testing the models
    errors = []
    for model in models:
        error = 0
        for i in range(k):
            error += fit(model, x_fold[not i], y_fold[not i],
                                x_fold[i], y_fold[i])
        error.append(error)

    #finding the index of the lowest-cost-model
    errors = np.array(errors)
    best_idx = np.argmin(errors)

    return best_idx, errors

In [9]:
# Define batch sizes to try
batch_sizes = np.arange(40,60,2)
batch_results = []

for bs in batch_sizes:
    tuner = keras_tuner.RandomSearch(
        hypermodel=build_model,
        objective="val_accuracy",
        max_trials=5,
        overwrite=True,
        directory="my_dir",
        project_name=f"batch_{bs}"
    )
    tuner.search(x_train, y_train,
                 epochs=200,
                 batch_size=bs,
                 validation_data=(x_valid, y_valid),
                 verbose=2)
    
    best_hp = tuner.get_best_hyperparameters(1)[0]
    best_acc = tuner.oracle.get_best_trials(1)[0].score
    batch_results.append({"batch_size": bs, "val_accuracy": best_acc})


batch = pd.DataFrame(batch_results)
print(batch)


Search: Running Trial #1

Value             |Best Value So Far |Hyperparameter
0.1               |0.1               |dropout_rate
relu              |relu              |activation
0.081613          |0.081613          |learning_rate
adam              |adam              |optimizer

Epoch 1/200
210/210 - 3s - 12ms/step - accuracy: 0.4974 - loss: 0.6992 - val_accuracy: 0.5200 - val_loss: 0.6924
Epoch 2/200
210/210 - 1s - 3ms/step - accuracy: 0.5040 - loss: 0.6963 - val_accuracy: 0.4800 - val_loss: 0.6935
Epoch 3/200
210/210 - 1s - 3ms/step - accuracy: 0.4946 - loss: 0.6948 - val_accuracy: 0.4800 - val_loss: 0.6984
Epoch 4/200
210/210 - 1s - 3ms/step - accuracy: 0.4992 - loss: 0.6953 - val_accuracy: 0.5200 - val_loss: 0.6925
Epoch 5/200
210/210 - 1s - 3ms/step - accuracy: 0.5018 - loss: 0.6948 - val_accuracy: 0.4800 - val_loss: 0.6945
Epoch 6/200
210/210 - 1s - 3ms/step - accuracy: 0.4964 - loss: 0.6947 - val_accuracy: 0.4800 - val_loss: 0.6932
Epoch 7/200


KeyboardInterrupt: 

In [10]:
# Get all trials
trials = tuner.oracle.trials

# Extract dropout rate and val_accuracy for each trial
results = []
for trial in trials.values():
    dropout = trial.hyperparameters.values["dropout_rate"]
    lr = trial.hyperparameters.values["learning_rate"]
    optimizer = trial.hyperparameters.values["optimizer"]
    val_acc = trial.score  # best val_accuracy for that trial
    results.append({"dropout_rate": dropout, "learning_rate": lr,
                    "optimizer": optimizer, "val_accuracy": val_acc})

# Convert to DataFrame
df = pd.DataFrame(results)
print(df)

   dropout_rate  learning_rate optimizer val_accuracy
0           0.1       0.081613      adam         None


In [11]:
import matplotlib.pyplot as plt

# Group by dropout and average val_accuracy
grouped = df.groupby("dropout")["val_accuracy"].mean()

plt.plot(grouped.index, grouped.values, marker='o')
plt.xlabel("Dropout Rate")
plt.ylabel("Val Accuracy")
plt.title("Accuracy vs Dropout Rate")
plt.grid(True)
plt.show()

KeyError: 'dropout'